In [1]:
# 卡关定义：用户重复进行某一关闯关，且失败
# 后续行为：付费、流失、继续活跃
# 数据来源：JP-8.1-9.30

In [1]:
import pandas as pd
import numpy as np

In [7]:
df_count= pd.read_csv('../临时文件/10月付费金额和次数.csv',index_col=False).query('Metric=="B:内购次数"')[['当前最大章节id','googleShopID','Total']]
df_pay= pd.read_csv('../临时文件/10月付费金额和次数.csv',index_col=False).query('Metric=="A:内购金额"')[['当前最大章节id','googleShopID','Total']]

In [51]:
# 商品对照表
shopinfo = pd.read_excel('../临时文件/商品对照.xlsx')

In [16]:
shopinfo.columns=['name','price','googleShopID']

In [17]:
df_count=df_count.merge(shopinfo[['googleShopID','name']],how='left',on='googleShopID')
df_pay=df_pay.merge(shopinfo[['googleShopID','name']],how='left',on='googleShopID')

In [18]:
# 创建数据透视表
df_count.pivot_table(values="Total",# 计算字段
                      index="name",# 分组的行
                      columns="当前最大章节id",# 分组的列
                      aggfunc="sum", # 聚合方式
                      fill_value=0, # 对缺失值的填充，默认为nan
                      # margins=True, # 是否启用总计
                      # margins_name='总计' # 总计的名称
                     ).to_excel('../临时文件/关卡付费礼包次数.xlsx')

In [19]:
# 创建数据透视表
df_pay.pivot_table(values="Total",# 计算字段
                      index="name",# 分组的行
                      columns="当前最大章节id",# 分组的列
                      aggfunc="sum", # 聚合方式
                      fill_value=0, # 对缺失值的填充，默认为nan
                      # margins=True, # 是否启用总计
                      # margins_name='总计' # 总计的名称
                     ).to_excel('../临时文件/关卡付费礼包金额.xlsx')

In [2]:
# 10月付费用户闯关数据
df = pd.read_csv('../临时文件/10月用户闯关数据.csv',index_col=False)[['用户唯一id','事件发生时间','level_id','是否通关']]

In [115]:
# 计算每一关的闯关人数
df.query('level_id>=20&level_id<=40').groupby(by='level_id',as_index=False)['用户唯一id'].nunique()


,level_id,用户唯一id
0,20,816
1,21,696
2,22,706
3,23,676
4,24,630
5,25,471
6,26,422
7,27,432
8,28,398
9,29,382


In [117]:
# 计算关卡的人均失败次数
df1=df.query('level_id>=20&level_id<=40&是否通关==0').groupby(by='level_id',as_index=False)[['用户唯一id','事件发生时间']].agg({
    '用户唯一id':'nunique',
    '事件发生时间':'count'})

In [118]:
df1['人均失败次数']=df1['事件发生时间']/df1['用户唯一id']

In [120]:
# 计算首次和最后一次关卡的时间
df['时间']=pd.to_datetime(df['事件发生时间'],unit='ms')
result = df.groupby(by=['用户唯一id', 'level_id'], as_index=False).agg(
    时间_min=('时间', 'min'),
    时间_max=('时间', 'max')
)# 不会产生聚合索引


In [123]:
result['卡关天数']=(result['时间_max'] - result['时间_min']).dt.days

In [126]:
result.query('level_id>=20&level_id<=40').groupby(by='level_id',as_index=False)[['用户唯一id','卡关天数']].agg({
    '用户唯一id':'nunique',
    '卡关天数':'sum'
})

,level_id,用户唯一id,卡关天数
0,20,816,2160
1,21,696,1010
2,22,706,1620
3,23,676,2501
4,24,630,2710
5,25,471,783
6,26,422,561
7,27,432,1224
8,28,398,1176
9,29,382,1048


In [4]:
df.head()

,用户唯一id,事件发生时间,level_id,是否通关
0,5GXb3DOZe7r5oucUUTQ0Db,1728074768738,11,0
1,4D9vWUSapMfFS1uAjdFxBu,1728958800011,11,1
2,7Fk17cilKombqhm8FxDI93,1729347433331,20,0
3,1Ty6MZIC10BZB3GXJNVpG8,1728572909303,13,1
4,1J5DZpm45yNKPxcdaFER6a,1727927613761,7,0


In [129]:
# 20-41关的漏斗
level_ids=[i for i in range(20,42)]
for i in level_ids:
    df[f'闯关第{i}关'] = df.apply(lambda x:x['用户唯一id'] if x['level_id'] == i else np.nan,axis=1)
df1 = df.groupby(by='用户唯一id')[[f'闯关第{i}关' for i in range(20,42)]].agg({f'闯关第{i}关':'nunique' for i in range(20,42)})

In [131]:
# 活跃数据
df_login = pd.read_csv('../临时文件/10月用户登录数据.csv',index_col=False)

In [133]:
# 用户最后一次登录
df_login = df_login.groupby(by='用户唯一id',as_index=False)['事件发生时间'].max()

In [134]:
df_login['最后登录时间'] = pd.to_datetime(df_login['事件发生时间'],unit='ms')

In [135]:
# 连接用户
res1=df1.merge(df_login[['用户唯一id','最后登录时间']],how='left',on='用户唯一id')

In [138]:
# 判断关卡流失
# 1 用户还未进入20-40关
# 2 用户进入20-40关，计算用户最后所在关卡，和最后登录时间是否距离今日超过2天
def func2(x):
    # 默认最后一次闯关 = 19
    last_level=19
    for  i in  range(20,42):
        if x[f'闯关第{i}关']==1 :
            last_level = i
    if last_level ==19:
        return '尚未进入20-40关'
    else:
        loss_day = loss_day = (pd.to_datetime('2024-10-23') - x['最后登录时间']).days
        if loss_day >2:
            return last_level
        else:
            return '未流失'

res1['流失关卡'] = res1.apply(lambda x:func2(x),axis=1)
    

In [140]:
res1.groupby(by='流失关卡',as_index=False)['用户唯一id'].nunique()

,流失关卡,用户唯一id
0,20,20
1,21,7
2,22,34
3,23,26
4,24,35
5,25,19
6,26,9
7,27,8
8,28,12
9,29,7


In [9]:
# 用户付费数据
df_ng=pd.read_csv('../临时文件/20-40关用户付费.csv',index_col=False)[['用户唯一id','事件发生时间','googleShopID','Total']]

In [11]:
df_ng['付费时间'] = pd.to_datetime(df_ng['事件发生时间'],unit='ms')

In [5]:
df['闯关时间'] = pd.to_datetime(df['事件发生时间'],unit='ms')

In [20]:
# 判断是否20-40关之间付费
# 计算用户闯关失败的时间
df_level_time=df.query('是否通关==0&level_id>=20&level_id<=40').pivot_table(values="闯关时间",# 计算字段
                      index="用户唯一id",# 分组的行
                      columns="level_id",# 分组的列
                      aggfunc="min", # 聚合方式
                      fill_value=0, # 对缺失值的填充，默认为nan
                      margins=False, # 是否启用总计
                      # margins_name='总计' # 总计的名称
                     )


In [21]:
df_level_time=df_level_time.merge(df_ng[['用户唯一id','付费时间','googleShopID','Total']],how='left',on='用户唯一id')

In [23]:
df_level_time.columns=['用户唯一id']+[f'{i}' for i in range(20,41)]+['付费时间', 'googleShopID','Total']

In [36]:
df_level_time.columns

Index(['用户唯一id', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29',
       '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40',
       '付费时间', 'googleShopID', 'Total'],
      dtype='object')

In [46]:
# 判断付费是否是在闯关失败后一天内进行
def func(x):
    if pd.isna(x['付费时间']):
        return '未付费用户'
    for i in range(20,41):
        if x[f'{i}']==0:
            continue
        if (x['付费时间']-x[f'{i}']).days <=1   and (x['付费时间']-x[f'{i}']).days >=0:
            print(x['付费时间'],x[f'{i}'])
            # print('卡关付费')
            return '卡关付费'
        else: 
            return '非卡关付费'

df_level_time['是否卡关付费'] = df_level_time.apply(lambda x:func(x),axis=1)


2024-10-22 14:18:22.434000 2024-10-22 14:08:47.568000
2024-10-06 05:51:01.361000 2024-10-06 05:32:13.856000
2024-10-14 02:44:03.469000 2024-10-13 03:16:35.652000
2024-10-14 02:16:43.025000 2024-10-13 03:16:35.652000
2024-10-19 07:17:20.609000 2024-10-19 06:38:30.309000
2024-10-19 06:50:02.287000 2024-10-19 06:38:30.309000
2024-10-12 19:42:01.094000 2024-10-12 06:41:12.079000
2024-10-22 10:27:59.934000 2024-10-22 10:27:42.132000
2024-10-08 16:18:34.441000 2024-10-08 16:01:49.262000
2024-10-22 04:53:40.160000 2024-10-20 09:53:24.332000
2024-10-13 04:25:34.639000 2024-10-12 16:00:39.971000
2024-10-14 12:15:14.180000 2024-10-12 16:00:39.971000
2024-10-13 04:41:13.220000 2024-10-12 16:00:39.971000
2024-10-14 04:54:04.507000 2024-10-12 16:00:39.971000
2024-10-13 09:21:42.531000 2024-10-12 16:00:39.971000
2024-10-14 04:43:26.853000 2024-10-12 16:00:39.971000
2024-10-19 01:51:33.585000 2024-10-19 00:25:30.600000
2024-10-19 01:44:40.776000 2024-10-19 00:25:30.600000
2024-10-19 01:46:04.201000 2

In [47]:
df_level_time.groupby(by='是否卡关付费',as_index=False)[['用户唯一id','Total']].agg({
    '用户唯一id':'nunique',
    'Total':'sum'
})

,是否卡关付费,用户唯一id,Total
0,卡关付费,192,4698.0
1,未付费用户,625,0.0
2,非卡关付费,508,28872.0


In [50]:
df_ng_info=df_level_time.query('是否卡关付费=="卡关付费"').groupby(by='googleShopID',as_index=False)[['付费时间','Total']].agg({
    '付费时间':'count','Total':'sum'})

In [53]:
shopinfo.columns=['name','price','googleShopID']

In [55]:
df_ng_info=df_ng_info.merge(shopinfo,how='left',on='googleShopID')

In [56]:
df_ng_info.to_excel('../临时文件/卡关主要购买礼包.xlsx',index=False)

In [104]:
df4=df3.merge(df[['用户唯一id','level_id']],how='left',on='用户唯一id')

In [106]:
df4.groupby(by='level_id_y')['用户唯一id'].nunique().to_csv('../临时文件/每一关活跃.csv')

In [79]:
df['闯关时间']=pd.to_datetime(df['事件发生时间'],unit='ms')

In [80]:
# 10月付费用户登录数据
df2 = pd.read_csv('../临时文件/10月付费用户登录.csv',index_col=False)[['用户唯一id','事件发生时间']]
df2['登录时间']=pd.to_datetime(df2['事件发生时间'],unit='ms')

In [81]:
# 计算用户每一关的最后一次闯关时间
df['关卡最后闯关时间'] = df.groupby(by=['用户唯一id','level_id'])['闯关时间'].transform('max')

In [82]:
# 前41关的漏斗
level_ids=[i for i in range(2,42)]
for i in level_ids:
    df[f'闯关第{i}关'] = df.apply(lambda x:x['用户唯一id'] if x['level_id'] == i else np.nan,axis=1)

In [83]:
df=df.groupby(by='用户唯一id')[[f'闯关第{i}关' for i in range(2,42)]].agg({f'闯关第{i}关':'nunique' for i in range(2,42)})

In [86]:
# 用户最后一次登录时间
df2=df2.groupby(by='用户唯一id',as_index=False)['登录时间'].max()

In [87]:
# 连接登录时间，判断用户具体流失关卡
# 1. 用户顺利进入到下一关
# 2. 用户没有进入下一关，且最后一次登录距离今日超过2日
res= df.merge(df2,how='left',on='用户唯一id')


In [88]:
# 流失判断：
def loss_func(x):
    # 最后关卡位置,默认第二关
    loss_level=2
    for  i in  range(2,42):
        if x[f'闯关第{i}关']==1 :
            loss_level = i 
    #判断最后一次登录时间距离今日的差距天数
    loss_day = (pd.to_datetime('2024-10-23') - x['登录时间']).days
    if loss_day <= 2:
        loss_level ='尚未流失'
    return loss_level

res['流失关卡'] = res.apply(lambda x:loss_func(x),axis=1)
res=res.query('闯关第2关==1')
    
        
    

In [93]:
res['用户唯一id'].nunique()

1229

In [95]:
res.query('流失关卡==3')[['用户唯一id']].to_excel('../临时文件/第三关流失付费用户.xlsx',index=False)

In [91]:
res.groupby(by='流失关卡',as_index=False)['用户唯一id'].nunique()

,流失关卡,用户唯一id
0,2,25
1,3,123
2,4,27
3,5,24
4,6,25
5,7,19
6,8,5
7,9,12
8,10,21
9,11,6


In [8]:
df_pay.head()

,当前最大章节id,googleShopID,Total
1,3.0,tales_no_ad_forever,1611
2,3.0,tales_award_2,1262
5,3.0,tales_award_3,560
7,3.0,tales_award_1,530
9,4.0,tales_super_monthly_card,480


In [111]:
# 闯关数据
df1 = pd.read_csv('../临时文件/9月用户卡关.csv',index_col=False)[['用户唯一id','事件发生时间','level_id']]

In [112]:
df1['时间']=pd.to_datetime(df1['事件发生时间'],unit='ms')

In [113]:
df1.head()

,用户唯一id,事件发生时间,level_id,时间
0,7dxolq9SaVx0DU7blY5VNu,1725381179503,7,2024-09-03 16:32:59.503
1,2RS8G4PzakcunMyzisreLS,1725593953395,27,2024-09-06 03:39:13.395
2,3AuAGaZS7rCFTDtnELHTaK,1725631177822,10,2024-09-06 13:59:37.822
3,1ZxEiZIVe5ftymmBTfkALD,1725177352087,1,2024-09-01 07:55:52.087
4,5RukvsaAQD3RRuIN15TZYn,1726012098177,38,2024-09-10 23:48:18.177


In [9]:
# 活跃数据
df2 = pd.read_csv('../临时文件/9月用户活跃.csv',index_col=False)[['用户唯一id','事件发生时间']]

In [10]:
df2['活跃时间']=pd.to_datetime(df2['事件发生时间'],unit='ms')

In [11]:
df2.head()

,用户唯一id,事件发生时间,活跃时间
0,3mkkQ9DhxUffDYGfiOIxrE,1727838743375,2024-10-02 03:12:23.375
1,13namfQBRu8wjF3yYoN6Xs,1727971245577,2024-10-03 16:00:45.577
2,4H05TEyXGn20wqmKsk5wm5,1727404974845,2024-09-27 02:42:54.845
3,4k8q0HaTe09NhDVteg1lqs,1727951639526,2024-10-03 10:33:59.526
4,1xLV8uRzEjPXty52SoqD0c,1727513663312,2024-09-28 08:54:23.312


In [12]:
# 付费数据
df3=pd.read_csv('../临时文件/9月用户付费.csv',index_col=False)[['用户唯一id','事件发生时间']]

In [13]:
df3['付费时间']=pd.to_datetime(df3['事件发生时间'],unit='ms')

In [14]:
df3.head()

,用户唯一id,事件发生时间,付费时间
0,IysmpfoRA2nGv32zzhPFB,1725283761308,2024-09-02 13:29:21.308
1,1cvDmwbp0MLZfezs9ficVt,1727783339449,2024-10-01 11:48:59.449
2,60lWO4tL1riCPmY1dRpCvd,1725621629625,2024-09-06 11:20:29.625
3,NNy3gzVWsriONSkPJaXie,1727830209890,2024-10-02 00:50:09.890
4,5XwOft7FU5gDwM0WHM2hNv,1727976857077,2024-10-03 17:34:17.077


In [114]:
# 计算用户最后一次卡关
df1['卡关顺序']=df1.groupby(by='用户唯一id')['level_id'].rank(ascending=False,method='dense')

In [167]:
# 计算每一关失败次数
res_stuck=df1.query('level_id>=20').groupby(by=['用户唯一id','level_id'],as_index=False)[['事件发生时间','时间']].agg({
    '事件发生时间':'count',
    '时间':'min'
})

In [173]:
res_stuck.columns=['用户唯一id','关卡id','失败次数','失败时间']

In [174]:
for i in range(20,31):
    col_name = f'卡{i}关时间'
    res_stuck[col_name] = res_stuck.apply(lambda x: x['失败时间'] if x['关卡id'] == i else np.nan,axis=1)

In [177]:
res_stuck=res_stuck.query('失败次数>3').groupby(by='用户唯一id',as_index=False)[[f'卡{i}关时间' for i in range(20,31)]].agg({
    f'卡{i}关时间':'max' for i in range(20,31)
})

In [178]:
# 卡关过程中是否付费
res_stuck_ng=res_stuck.merge(df_ng,how='left',on='用户唯一id')

In [181]:
def func1(x):
    time_list =[x[f'卡{i}关时间'] for i in range(20,31)]
    
    if x['付费时间'] >min(time_list) and x['付费时间']< max(time_list):
        return '是'
    else:
        return '否'

In [184]:
res_stuck_ng['是否卡关付费'] = res_stuck_ng.apply(lambda x:func1(x),axis=1)

In [187]:
res_stuck_ng['用户唯一id'].nunique()

631

In [186]:
# 计算卡每一关时间的
res_stuck_ng.groupby(by = '是否卡关付费')[['用户唯一id']].nunique()


,用户唯一id
是否卡关付费,
否,626
是,16


In [188]:
level_df.head()

,用户唯一id,卡关关卡,失败次数,失败时间
0,102PihDDyyWvmxweVcCtLt,3,1,2024-09-04 02:02:26.375
1,103JTlgz3Bh2LZhuXlNI0y,10,4,2024-09-28 23:29:24.925
2,10cFSByYSnO8b3fPo2gPhP,5,1,2024-09-10 06:18:16.153
3,10hi1v1ne16RCC8Orxnmxm,31,24,2024-09-23 21:27:10.020
4,10kFoJsdHtEcLLtBbKs6hn,35,20,2024-09-01 01:05:28.780


In [198]:
df1_new=df1[df1['卡关顺序']==1].groupby(by=['用户唯一id','level_id'],as_index=False).agg(
    时间_min=('时间', 'min'),
    时间_max=('时间', 'max')
)

In [201]:
df1_new['卡关天数'] = (df1_new['时间_max'] - df1_new['时间_min']).dt.days

In [204]:
df1_new.query('level_id>=20').groupby(by='level_id',as_index=False)['卡关天数'].sum()

,level_id,卡关天数
0,20,94
1,21,27
2,22,195
3,23,334
4,24,578
5,25,119
6,26,15
7,27,176
8,28,177
9,29,70


In [116]:
level_df = df1[df1['卡关顺序']==1].groupby(by=['用户唯一id','level_id'],as_index=False)[['卡关顺序','时间']].agg({
    '卡关顺序':'count',
    '时间':'min'}
)

In [117]:
level_df.columns=['用户唯一id','卡关关卡','失败次数','失败时间']

In [122]:
df_ng = pd.read_csv('../临时文件/卡关用户付费.csv',index_col=False)[['用户唯一id','事件发生时间','Total']]

In [124]:
df_ng['付费时间']=pd.to_datetime(df_ng['事件发生时间'],unit='ms')

In [126]:
res_ng=level_df.query('卡关关卡>=20').merge(df_ng[['用户唯一id','付费时间','Total']],how='left',on='用户唯一id')

In [129]:
# 卡关前后付费对比
res_ng=res_ng[res_ng['Total'].notnull()]

In [131]:
res_ng['卡关前后'] = res_ng.apply(lambda x:'卡关后' if x['付费时间']>x['失败时间'] else'卡关前',axis=1)

C:\Users\admin\AppData\Local\Temp\ipykernel_17384\2880005215.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_ng['卡关前后'] = res_ng.apply(lambda x:'卡关后' if x['付费时间']>x['失败时间'] else'卡关前',axis=1)


In [135]:
res_ng['用户唯一id'].nunique()

267

In [134]:
res_ng.groupby(by='卡关前后')[['用户唯一id','Total']].agg({
    '用户唯一id':'nunique',
    'Total':'sum'
})

,用户唯一id,Total
卡关前后,,
卡关前,261,17419.0
卡关后,73,1766.0


In [136]:
level_df.query('卡关关卡>=20')['用户唯一id'].nunique()

758

In [137]:
# 卡关后流失，统一生命周期
df_user = level_df.query('卡关关卡>=20')


In [138]:
res_loss = df_user.merge(df2,how='left',on='用户唯一id')

In [140]:
res_loss['生命周期']=(res_loss['活跃时间'] - res_loss['失败时间']).dt.days

In [143]:
res_loss.query('生命周期>0').groupby(by='生命周期',as_index=False)['用户唯一id'].nunique()

,生命周期,用户唯一id
0,1,574
1,2,535
2,3,504
3,4,412
4,5,360
5,6,313
6,7,274
7,8,264
8,9,218
9,10,195


In [145]:
res_pay = df_user.merge(df_ng,how='left',on='用户唯一id')
res_pay = res_pay[res_pay['付费时间'].notnull()]
res_pay['生命周期']=(res_pay['付费时间'] - res_pay['失败时间']).dt.days


In [146]:
res_pay.query('生命周期>=0').groupby(by='生命周期',as_index=False)[['用户唯一id','Total']].agg({
    '用户唯一id':'nunique',
    'Total':'sum'
})

,生命周期,用户唯一id,Total
0,0,26,343.0
1,1,17,349.0
2,2,16,130.0
3,3,17,185.0
4,4,11,90.0
5,5,7,56.0
6,6,7,123.0
7,7,7,38.0
8,8,5,49.0
9,9,5,165.0


In [119]:
level_df.query('卡关关卡>=20').groupby(by='卡关关卡')[['用户唯一id','失败次数']].agg({
    '用户唯一id':'nunique',
    '失败次数':'sum'
})

,用户唯一id,失败次数
卡关关卡,,
20,40,184
21,20,60
22,50,361
23,76,744
24,113,793
25,35,418
26,9,38
27,32,331
28,47,688


In [75]:
# 卡关后付费
level_df= level_df.merge(df3,how='left',on='用户唯一id')

In [76]:
level_df['卡关后付费']=level_df.apply(lambda x:'是' if x['付费时间']>x['失败时间'] else '否',axis=1)

In [78]:
level_df.groupby(by='卡关后付费',as_index=False)['用户唯一id'].nunique()

,卡关后付费,用户唯一id
0,否,2761
1,是,176


In [79]:
level_df['用户唯一id'].nunique()

2837

In [80]:
# 卡关后流失
level_df2 = df1[df1['卡关顺序']==1].groupby(by=['用户唯一id','level_id'],as_index=False)[['卡关顺序','时间']].agg({
    '卡关顺序':'count',
    '时间':'max'}
)


In [81]:
level_df2.columns=['用户唯一id','卡关关卡','失败次数','失败时间']

In [82]:
# 卡关后流失
level_df2= level_df2.merge(df2,how='left',on='用户唯一id')

In [92]:
level_df2['活跃时间差'] = (level_df2['活跃时间'] - level_df2['失败时间']).dt.days

In [108]:
level_df2['是否流失'] = level_df2.apply(lambda x: '否' if x['活跃时间差']>=1 else '是',axis=1)

In [109]:
level_df2.groupby(by='是否流失',as_index=False)['用户唯一id'].nunique()

,是否流失,用户唯一id
0,否,1324
1,是,2836


In [110]:
level_df2['用户唯一id'].nunique()

2837

In [149]:
df4 = pd.read_csv('../临时文件/卡关用户挑战.csv',index_col=False)[['change_name','用户唯一id','事件发生时间']]

In [150]:
df4['挑战时间']=pd.to_datetime(df4['事件发生时间'],unit='ms')

In [151]:
res_chall=level_df.query('卡关关卡>=20').merge(df4,how='left',on='用户唯一id')

In [153]:
res_chall.query('挑战时间>失败时间').groupby(by='change_name',as_index=False)['用户唯一id'].nunique()

,change_name,用户唯一id
0,gold,180
1,tower,504


In [158]:
df5 =pd.read_csv('../临时文件/卡关用户废墟探索.csv',index_col=False)[['用户唯一id','事件发生时间']]

In [159]:
df5['探索时间'] = pd.to_datetime(df5['事件发生时间'],unit='ms')

In [161]:
res_exp=level_df.query('卡关关卡>=23').merge(df5[['用户唯一id','探索时间']],how='left',on='用户唯一id')

In [163]:
res_exp.query('探索时间>失败时间')['用户唯一id'].nunique()

322

In [164]:
res_exp['用户唯一id'].nunique()

648